# Acme Robotics walkthrough, notebook 1: ontology lifecycle & governance

A runnable companion to `docs/ACME_ROBOTICS_WALKTHROUGH.md` sections 2-6 --
every code cell below actually executes, against the real worked example in
`examples/acme_robotics/` (a small "Acme Robotics" org chart built on the
real W3C Organization Ontology and FOAF vocabularies, fetched from
`https://www.w3.org/ns/org#` and `http://xmlns.com/foaf/spec/index.rdf` and
committed under `examples/acme_robotics/reference_vocab/` so this notebook
runs offline). Markdown cells stay short and link back to the walkthrough
doc for full narrative depth; this notebook's job is to *show it actually
working*.

See `docs/acme_robotics_data_pipeline.ipynb` for the CSV-to-RDF pipeline and
live-triplestore checking (walkthrough doc sections 7-9).

## 0. Setup

In [1]:
import subprocess, sys, tempfile, shutil, os
from pathlib import Path
import pandas as pd

def _find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError("could not find the repo root (looked for pyproject.toml + examples/) above " + str(start))

REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
EXAMPLE = "examples/acme_robotics"
assert (REPO_ROOT / EXAMPLE / "acme-org-v1.ttl").is_file(), \
    f"expected {EXAMPLE}/acme-org-v1.ttl under {REPO_ROOT}"
print("repo root:", REPO_ROOT)

LAST_EXIT_CODE = None

def run(*args, cwd=REPO_ROOT, check=True):
    """Runs the ontology-quality-suite CLI and prints its stdout. Returns
    nothing (keeps Jupyter's REPL-style auto-display from dumping a raw
    return value under every call) -- the exit code, when needed (the
    CI-gate demo below), is available afterwards as LAST_EXIT_CODE."""
    global LAST_EXIT_CODE
    proc = subprocess.run(["ontology-quality-suite", *args], cwd=cwd, capture_output=True, text=True)
    LAST_EXIT_CODE = proc.returncode
    print(proc.stdout)
    if proc.returncode != 0 and proc.stderr:
        print(proc.stderr, file=sys.stderr)
    if check and proc.returncode not in (0, 1):  # 1 = findings triggered --fail-on, still a clean run
        raise RuntimeError(f"exit {proc.returncode}")

def summary(out_dir):
    """Reads full_results.csv and returns a per-check-id/severity finding count Series."""
    df = pd.read_csv(Path(out_dir) / "full_results.csv")
    return df.groupby(["check_id", "severity"]).size()

repo root: C:\repos\consolidated_ontology_suite_python


## 1. Ontology-only quality gate (`ontology`)

Walkthrough doc §2. No data, just "is the TBox itself sound" -- runs the
always-on `owlrl` closure/pattern checks and, if `uv sync --extra reasoner`
is installed with a working Java/HermiT setup, a real external DL reasoner
too. `LOG-001` (Contractor disjoint with its own ancestor Employee) is the
one deliberate contradiction in this fixture and fires from the always-on
pass alone; a working external reasoner independently confirms it twice
more (`REA-020`/`REA-021`). HermiT is occasionally environment-flaky in
ways unrelated to this fixture -- if that happens here, you'll see a
`REA-022` note instead, which is the suite degrading gracefully rather than
failing the run (see `docs/REASONING.md`).

In [2]:
run("ontology", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl",
    "--import-dir", f"{EXAMPLE}/reference_vocab", "--out-dir", "out/nb1-ontology", "--fail-on", "never")
print(summary("out/nb1-ontology"))

Findings: 3 total (3 Violation, 0 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb1-ontology
  - out\nb1-ontology\report.html (start here)
  - out\nb1-ontology\full_results.csv

check_id  severity 
LOG-001   Violation    1
REA-020   Violation    1
REA-021   Violation    1
dtype: int64


## 2. The full registry, and "whose problem is this?" (`checks`)

Walkthrough doc §3. Run with the real `org:`/`foaf:` imports resolved,
`checks` reports a lot of findings -- most of them pre-existing
documentation/style gaps *inside those upstream vocabularies themselves*,
not Acme's own additions.

In [3]:
run("checks", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--import-dir", f"{EXAMPLE}/reference_vocab",
    "--engine", "sparql", "--out-dir", "out/nb1-checks-all", "--fail-on", "never")
all_findings = summary("out/nb1-checks-all")
print("total findings (incl. upstream org:/foaf:):", all_findings.sum())

Findings: 294 total (48 Violation, 162 Warning, 84 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb1-checks-all
  - out\nb1-checks-all\report.html (start here)
  - out\nb1-checks-all\full_results.csv

total findings (incl. upstream org:/foaf:): 294


`--exclude-imports` narrows this, but reintroduces its own noise -- with
FOAF's own triples gone, references *to* `foaf:Person` etc. become
"unresolved reference target" findings instead:

In [4]:
run("checks", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--exclude-imports",
    "--engine", "sparql", "--out-dir", "out/nb1-checks-excluded", "--fail-on", "never")
excluded_findings = summary("out/nb1-checks-excluded")
print("findings with --exclude-imports (includes its own reference-target noise):", excluded_findings.sum())

Findings: 9 total (1 Violation, 8 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb1-checks-excluded
  - out\nb1-checks-excluded\report.html (start here)
  - out\nb1-checks-excluded\full_results.csv

findings with --exclude-imports (includes its own reference-target noise): 9


`--own-namespace` is the precise fix: keep imports fully resolved (so a
real domain/range check against an imported property still fires if it
should) and filter the *report* to Acme's own IRI prefix instead. This
gives exactly the four deliberate flaws from §1's table (`LOG-001`,
`QUA-001`/`QUA-004`, `STR-003`, `STY-002`), nothing upstream and nothing
from the exclude-imports noise:

In [5]:
run("checks", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--import-dir", f"{EXAMPLE}/reference_vocab",
    "--engine", "sparql", "--own-namespace", "https://acme.example.org/",
    "--out-dir", "out/nb1-checks-own", "--fail-on", "never")
own_findings = summary("out/nb1-checks-own")
print("Acme's own findings (imports still resolved):", own_findings.sum())
own_findings

Findings: 5 total (1 Violation, 4 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb1-checks-own
  - out\nb1-checks-own\report.html (start here)
  - out\nb1-checks-own\full_results.csv

Acme's own findings (imports still resolved): 5


check_id  severity 
LOG-001   Violation    1
QUA-001   Warning      1
QUA-004   Warning      1
STR-003   Warning      1
STY-002   Warning      1
dtype: int64

## 3. A project-specific check (`--registry`/`--shapes`/`--sparql`)

Walkthrough doc §4. `examples/acme_robotics/custom_checks/` is a minimal,
project-local registry (one check, `ACM-001`: every `acme:Employee` must
carry an `acme:hasEmployeeId`) added per `docs/EXTENDING.md`, with zero
changes to the installed package. Demonstrated here against a deliberately
incomplete record; `docs/acme_robotics_data_pipeline.ipynb` additionally
confirms zero false positives against the real triplified employee data.

In [6]:
broken_dir = Path(tempfile.mkdtemp())
(broken_dir / "broken_employee.ttl").write_text('''
@prefix acme: <https://acme.example.org/ns/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
<https://acme.example.org/data/employee/E999> a acme:Employee ;
    foaf:name "No ID Given" .
''', encoding="utf-8")

run("checks", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--data", str(broken_dir / "broken_employee.ttl"),
    "--registry", f"{EXAMPLE}/custom_checks/registry.json",
    "--sparql", f"{EXAMPLE}/custom_checks/sparql",
    "--engine", "sparql", "--out-dir", "out/nb1-acm001", "--fail-on", "never")
print(summary("out/nb1-acm001"))

Findings: 1 total (1 Violation, 0 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb1-acm001
  - out\nb1-acm001\report.html (start here)
  - out\nb1-acm001\full_results.csv

check_id  severity 
ACM-001   Violation    1
dtype: int64


## 4. Reference documentation (`docgen`)

In [7]:
run("docgen", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--out-dir", "out/nb1-docgen")
print("Open out/nb1-docgen/ontology-documentation.html to view it.")

Wrote out\nb1-docgen\ontology_doc_data.json: prefix='acme', 4 classes, 2 object properties, 3 datatype properties, 1 sections, 5 external terms (0 resolved).
Wrote out\nb1-docgen\ontology-documentation.html (4 classes, 2 object properties, 3 datatype properties, 5 external terms).
Reference documentation written to: out\nb1-docgen\ontology-documentation.html
4 class diagram(s) written to: out\nb1-docgen\class-diagrams

Open out/nb1-docgen/ontology-documentation.html to view it.


## 5. CI gate behaviour (`--fail-on`)

Walkthrough doc §2/§3's same commands, but exiting non-zero on `Violation`
-- a green/red build gate needs nothing more exotic than `--fail-on`.

In [8]:
run("ontology", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--import-dir", f"{EXAMPLE}/reference_vocab",
    "--out-dir", "out/nb1-ci-gate", "--fail-on", "Violation", check=False)
print("exit code:", LAST_EXIT_CODE, "(1 = a Violation-severity finding exists -- correct, this ontology has one on purpose)")

Findings: 3 total (3 Violation, 0 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb1-ci-gate
  - out\nb1-ci-gate\report.html (start here)
  - out\nb1-ci-gate\full_results.csv

exit code: 1 (1 = a Violation-severity finding exists -- correct, this ontology has one on purpose)


* Owlready2 * WARNING: DataProperty http://xmlns.com/foaf/0.1/name belongs to more than one entity types: [owl.DatatypeProperty, rdf-schema.label]; I'm trying to fix it...
* Owlready2 * Running HermiT...
    java -Xmx2000M -cp C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/Pedro/AppData/Local/Temp/tmpl566mfiz -Y
* Owlready2 * HermiT took 1.6285557746887207 seconds
* Owlready * Reparenting C:\Users\Pedro\AppData\Local\Temp\tmpeetve2a6\ontology.homepage: {C:\Users\Pedro\AppData\Local\Temp\tmpeetve2a6\ontology.isPrimaryTopicOf, owl.ObjectProperty, C:\Users\Pedro\AppData\Local\Temp\tmpeetve2a6\ontology.page, owl.InverseFunctionalProperty} => {C:\Users\Pedro\AppData\Local\Temp\tmpeetve2a6\ontology.isPrimaryTopicOf}
* Owlready * Reparenting C:\Users\Pedro\AppData\Local\Temp\tmpeetve2a6\ontolog

## 6. The taxonomy boundary (`pattern-consistency`)

Walkthrough doc §8. `employees.csv` has a row whose department (`MKT`)
isn't declared in `taxonomy.ttl`'s controlled list. This static check
(query + ontology + taxonomy, no real data yet) reports **clean** --
correctly, not a false negative: it looks for a taxonomy value **hard-coded
in the query template's text**, and `employees.rq` builds its department
reference dynamically per CSV row. `docs/acme_robotics_data_pipeline.ipynb`
closes this exact gap once real triplified data is available
(`--output-data`, `pattern_consistency.check_taxonomy_membership`).

In [9]:
run("pattern-consistency",
    "--queries", f"{EXAMPLE}/employees.rq", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl",
    "--ontology", f"{EXAMPLE}/reference_vocab/org.ttl", "--ontology", f"{EXAMPLE}/reference_vocab/foaf.rdf",
    "--taxonomy", f"{EXAMPLE}/taxonomy.ttl", "--out-dir", "out/nb1-pattern-consistency")

No modelling-pattern inconsistencies found across ontology, taxonomy, transformation, or output data.

Written to: out\nb1-pattern-consistency\pattern-consistency.txt



## 7. Versioning and drift repair (`version-diff`, `consistency`)

Walkthrough doc §6. `acme-org-v2.ttl` renames `acme:Engineer` to
`acme:SoftwareEngineer` (a breaking change, with a migration annotation)
and adds two purely-additive terms.

In [10]:
run("version-diff", f"{EXAMPLE}/acme-org-v1.ttl", f"{EXAMPLE}/acme-org-v2.ttl",
    "--out-dir", "out/nb1-version-diff", "--json")

Ontology version diff: examples/acme_robotics/acme-org-v1.ttl -> examples/acme_robotics/acme-org-v2.ttl

Removed classes [MAJOR]:
  - https://acme.example.org/ns/Engineer

Removed subclass edges [MAJOR]:
  - https://acme.example.org/ns/Engineer no longer rdfs:subClassOf https://acme.example.org/ns/Employee

Added classes [minor]:
  - https://acme.example.org/ns/ProductManager
  - https://acme.example.org/ns/SoftwareEngineer

Added properties [minor]:
  - https://acme.example.org/ns/hireDate

Added subclass edges [minor]:
  - https://acme.example.org/ns/ProductManager rdfs:subClassOf https://acme.example.org/ns/Employee
  - https://acme.example.org/ns/SoftwareEngineer rdfs:subClassOf https://acme.example.org/ns/Employee

Added equivalentClass axioms [minor]:
  - https://acme.example.org/ns/Engineer equivalentClass https://acme.example.org/ns/SoftwareEngineer

Suggested version bump: MAJOR

Written to: out\nb1-version-diff\diff.txt
Written to: out\nb1-version-diff\diff.json



`employees.rq` types every employee `acme:Engineer` -- exactly the renamed
term. `consistency` detects this and proposes (and, with `--apply-repairs`,
applies) a rename fix. Note that only `--import-dir` is needed below, not a
repeatable `--ontology <path>` pointing at `org.ttl`/`foaf.rdf` by hand --
`consistency` resolves `--new`'s `owl:imports` for the TARQL-alignment half
the same way it always did for the version-diff half (see
`docs/CONSISTENCY_AND_REPAIR.md`).

In [11]:
run("consistency", "--new", f"{EXAMPLE}/acme-org-v2.ttl", "--old", f"{EXAMPLE}/acme-org-v1.ttl",
    "--queries", EXAMPLE, "--import-dir", f"{EXAMPLE}/reference_vocab",
    "--out-dir", "out/nb1-consistency")

Ontology version diff: examples/acme_robotics/acme-org-v1.ttl -> examples/acme_robotics/acme-org-v2.ttl

Removed classes [MAJOR]:
  - https://acme.example.org/ns/Engineer

Removed subclass edges [MAJOR]:
  - https://acme.example.org/ns/Engineer no longer rdfs:subClassOf https://acme.example.org/ns/Employee

Added classes [minor]:
  - https://acme.example.org/ns/ProductManager
  - https://acme.example.org/ns/SoftwareEngineer

Added properties [minor]:
  - https://acme.example.org/ns/hireDate

Added subclass edges [minor]:
  - https://acme.example.org/ns/ProductManager rdfs:subClassOf https://acme.example.org/ns/Employee
  - https://acme.example.org/ns/SoftwareEngineer rdfs:subClassOf https://acme.example.org/ns/Employee

Added equivalentClass axioms [minor]:
  - https://acme.example.org/ns/Engineer equivalentClass https://acme.example.org/ns/SoftwareEngineer

Suggested version bump: MAJOR

1 class(es) used in TARQL but not declared in the ontology set:
  [undeclared_class] https://acme.

In [12]:
# Apply it for real, on a throwaway copy -- never mutate the checked-in example.
scratch = Path(tempfile.mkdtemp())
shutil.copy(REPO_ROOT / EXAMPLE / "employees.rq", scratch / "employees.rq")

run("consistency", "--new", f"{EXAMPLE}/acme-org-v2.ttl", "--old", f"{EXAMPLE}/acme-org-v1.ttl",
    "--queries", str(scratch), "--import-dir", f"{EXAMPLE}/reference_vocab",
    "--out-dir", "out/nb1-consistency-apply", "--apply-repairs", "--min-confidence", "0.7")

rewritten = (scratch / "employees.rq").read_text(encoding="utf-8")
assert "SoftwareEngineer" in rewritten and "a acme:Employee, acme:Engineer" not in rewritten
print("employees.rq correctly rewritten to use acme:SoftwareEngineer.")

Ontology version diff: examples/acme_robotics/acme-org-v1.ttl -> examples/acme_robotics/acme-org-v2.ttl

Removed classes [MAJOR]:
  - https://acme.example.org/ns/Engineer

Removed subclass edges [MAJOR]:
  - https://acme.example.org/ns/Engineer no longer rdfs:subClassOf https://acme.example.org/ns/Employee

Added classes [minor]:
  - https://acme.example.org/ns/ProductManager
  - https://acme.example.org/ns/SoftwareEngineer

Added properties [minor]:
  - https://acme.example.org/ns/hireDate

Added subclass edges [minor]:
  - https://acme.example.org/ns/ProductManager rdfs:subClassOf https://acme.example.org/ns/Employee
  - https://acme.example.org/ns/SoftwareEngineer rdfs:subClassOf https://acme.example.org/ns/Employee

Added equivalentClass axioms [minor]:
  - https://acme.example.org/ns/Engineer equivalentClass https://acme.example.org/ns/SoftwareEngineer

Suggested version bump: MAJOR

1 class(es) used in TARQL but not declared in the ontology set:
  [undeclared_class] https://acme.

### The migration annotation works in either direction

`owl:equivalentClass` is logically symmetric -- a reasoner treats either
direction identically. This suite's rename-detection heuristic recognizes
*both* directions at full confidence: asserted from the old (removed) IRI
to the new one (the convention `acme-org-v2.ttl` uses above), or the other
way round (asserting it from the new term -- "here's what this replaces" --
is at least as natural to write). Verified directly below, against the same
Engineer/SoftwareEngineer rename this fixture uses. `dcterms:isReplacedBy`,
unlike `owl:equivalentClass`, is directional by definition and stays
one-way -- see `ontology_suite/versioning/rename_detection.py`'s module
docstring.

In [13]:
import rdflib
from ontology_suite.versioning.diff import diff_ontologies
from ontology_suite.versioning.rename_detection import detect_renames

def rename_confidence(migration_triple):
    old = rdflib.Graph()
    old.parse(data='''
        @prefix ex: <https://example.org/demo/> .
        @prefix owl: <http://www.w3.org/2002/07/owl#> .
        ex:Engineer a owl:Class .
    ''', format="turtle")
    new = rdflib.Graph()
    new.parse(data=f'''
        @prefix ex: <https://example.org/demo/> .
        @prefix owl: <http://www.w3.org/2002/07/owl#> .
        ex:SoftwareEngineer a owl:Class .
        {migration_triple}
    ''', format="turtle")
    diff, _bump = diff_ontologies(old, new)
    renames = detect_renames(diff, new)
    return renames[0].confidence if renames else None

forward = rename_confidence("ex:Engineer owl:equivalentClass ex:SoftwareEngineer .")   # old -> new
reverse = rename_confidence("ex:SoftwareEngineer owl:equivalentClass ex:Engineer .")   # new -> old
print("old -> new direction confidence:", forward)
print("new -> old direction confidence:", reverse)
assert forward == 1.0 and reverse == 1.0
print("Both directions recognized at full confidence.")

old -> new direction confidence: 1.0
new -> old direction confidence: 1.0
Both directions recognized at full confidence.


---
## Next

- `docs/acme_robotics_data_pipeline.ipynb` covers the CSV-to-RDF pipeline,
  the taxonomy-membership check against real triplified data, and
  live-triplestore checking (walkthrough doc §7-§9).
- `docs/ACME_ROBOTICS_WALKTHROUGH.md` has the full narrative behind every
  command above, plus pointers to `PRIMER.md`, `ARCHITECTURE.md`,
  `EXTENDING.md`, `CONSISTENCY_AND_REPAIR.md`, and
  `MODELLING_PATTERN_CONSISTENCY.md` for depth on any one topic.